In [7]:
# treat an auction order as [price, quantity, time]
# for time, 0 is them and 1 is us (after)
def sort_bids(bids: list[list[int]]):
    bids.sort(key = lambda x: [x[0], -x[2]])
    return bids
def sort_asks(asks: list[list[int]]):
    asks.sort(key = lambda x: [-x[0], -x[2]])
    return asks

In [8]:
# find the clearing price that maximizes volume
# break ties w/ highest clearing price possible
def find_clearing_price(bids: list[list[int]], asks: list[list[int]]):
    bids = sort_bids(bids)
    asks = sort_asks(asks)
    best_clearing_price = asks[-1][0] - 1
    best_volume = 0

    for price in range(asks[-1][0], bids[-1][0] + 1):
        bid_volume = sum([x[1] for x in bids if x[0] >= price])
        ask_volume = sum([x[1] for x in asks if x[0] <= price])
        volume = min(bid_volume, ask_volume)
        if volume >= best_volume: # higher prices w/ same volume will overwrite
            best_clearing_price = price
            best_volume = volume
    return best_clearing_price

In [9]:
# find clearing price, resolve bids + asks, compute profit
# add a variable for deductions on the second orderbook
def execute_market(bids: list[list[int]], asks: list[list[int]], breakeven: int, trade_penalty: float):
    bids = sort_bids(bids)
    asks = sort_asks(asks)
    clearing_price = find_clearing_price(bids, asks)
    
    copy_bids = [x for x in bids if x[0] >= clearing_price]
    copy_asks = [x for x in asks if x[0] <= clearing_price]
    profit = 0
    while len(copy_bids) > 0 and len(copy_asks) > 0 and copy_bids[-1][0] >= copy_asks[-1][0]:
        quantity = min(copy_bids[-1][1], copy_asks[-1][1])
        copy_bids[-1][1] -= quantity
        copy_asks[-1][1] -= quantity
        
        # handle our profit (time == 1)
        if copy_bids[-1][2] == 1:
            profit += (breakeven - clearing_price - trade_penalty) * quantity
        elif copy_asks[-1][2] == 1:
            profit += (clearing_price - breakeven - trade_penalty) * quantity
        
        # now delete orders!
        if copy_bids[-1][1] == 0:
            copy_bids.pop()
        if copy_asks[-1][1] == 0:
            copy_asks.pop()
    
    return profit

In [17]:
# brute force check for best order to send
from copy import deepcopy

def brute_force_check(bids: list[list[int]], asks: list[list[int]], breakeven: int, trade_penalty: float):
    min_price = min(min([x[0] for x in bids]), min([x[0] for x in asks])) - 1
    max_price = max(max([x[0] for x in bids]), max([x[0] for x in asks])) + 1
    
    # check bids
    best_bid_price = 0
    best_bid_volume = 0
    best_bid_profit = 0
    for price in range(min_price, max_price + 1):
        for volume in range(1, 75001):
            copy_bids = deepcopy(bids)
            copy_asks = deepcopy(asks)
            new_bid = [price, volume, 1]
            copy_bids.append(new_bid)
            
            cur_profit = execute_market(copy_bids, copy_asks, breakeven, trade_penalty)
            if cur_profit > best_bid_profit:
                best_bid_price = price
                best_bid_volume = volume
                best_bid_profit = cur_profit

    # check asks
    best_ask_price = 0
    best_ask_volume = 0
    best_ask_profit = 0
    for price in range(min_price, max_price + 1):
        for volume in range(1, 75001):
            copy_bids = deepcopy(bids)
            copy_asks = deepcopy(asks)
            new_ask = [price, volume, 1]
            copy_asks.append(new_ask)

            cur_profit = execute_market(copy_bids, copy_asks, breakeven, trade_penalty)
            if cur_profit > best_ask_profit:
                best_ask_price = price
                best_ask_volume = volume
                best_ask_profit = cur_profit

    if best_bid_profit > best_ask_profit:
        print("bid stuff:", best_bid_price, best_bid_volume, best_bid_profit)
        return best_bid_price, best_bid_volume, best_bid_profit
    elif best_bid_profit < best_ask_profit:
        print("ask stuff:", best_ask_price, best_ask_volume, best_ask_profit)
        return best_ask_price, best_ask_volume, best_ask_profit
    else:
        print("we have a dilemma here... return bid stuff")
        print("bid stuff:", best_bid_price, best_bid_volume, best_bid_profit)
        print("ask stuff:", best_ask_price, best_ask_volume, best_ask_profit)
        return best_bid_price, best_bid_volume, best_bid_profit

In [18]:
# first orderbook
bids = [
    [30, 30000, 0], 
    [29,  5000, 0], 
    [28, 12000, 0], 
    [27, 28000, 0],
]
asks = [
    [28, 40000, 0], 
    [31, 20000, 0], 
    [32, 20000, 0], 
    [33, 30000, 0],
]
breakeven = 30
trade_penalty = 0
price, volume, profit = brute_force_check(bids, asks, breakeven, trade_penalty)

bid stuff: 30 9999 9999


In [19]:
# second orderbook - we WILL be buying b/c market participants undervalue it
bids = [
    [20, 43000, 0], 
    [19, 17000, 0], 
    [18,  6000, 0], 
    [17,  5000, 0], 
    [16, 10000, 0], 
    [15,  5000, 0], 
    [14, 10000, 0], 
    [13,  7000, 0],
]
asks = [
    [12, 20000, 0],
    [13, 25000, 0],
    [14, 35000, 0],
    [15,  6000, 0],
    [16,  5000, 0],
    [18, 10000, 0],
    [19, 12000, 0],
]
breakeven = 20
trade_penalty = 0.1
price, volume, profit = brute_force_check(bids, asks, breakeven, trade_penalty)

bid stuff: 17 19999 77996.1
